### RAGto  Pipeline - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [11]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} Pdf files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            if len(documents) > 1:
                print(f" ✅ Loaded {len(documents)} pages")
            else: 
                print(f" ✅ Loaded {len(documents)} page")

        except Exception as e:
            print(f" ❌ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
all_pdf_documents = process_all_pdfs("../data")

Found 4 Pdf files to process

Processing: a-practical-guide-to-building-agents.pdf
 ✅ Loaded 34 pages

Processing: Definite integrals.pdf
 ✅ Loaded 10 pages

Processing: Subhamay_Dey_Resume.pdf
 ✅ Loaded 1 page

Processing: term 2 March 2026 .pdf
 ✅ Loaded 1 page

Total documents loaded: 46


In [12]:
all_pdf_documents

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-04-07T14:20:51+00:00', 'moddate': '2025-04-07T14:20:54+00:00', 'source': '..\\data\\pdf\\a-practical-guide-to-building-agents.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1', 'source_file': 'a-practical-guide-to-building-agents.pdf', 'file_type': 'pdf'}, page_content='A practical \u2028\nguide to \u2028\nbuilding agents'),
 Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-04-07T14:20:51+00:00', 'moddate': '2025-04-07T14:20:54+00:00', 'source': '..\\data\\pdf\\a-practical-guide-to-building-agents.pdf', 'total_pages': 34, 'page': 1, 'page_label': '2', 'source_file': 'a-practical-guide-to-building-agents.pdf', 'file_type': 'pdf'}, page_content='C o n t e n t s\nWha t is an agen t? 4\nWhen should y ou build an agen t


### Splitting into chunks

In [15]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for brtter RAG performance"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],  # \n\n - pararaph, \n - line, " " - space, "" - character
    )
    split_docs = splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs
chunks=split_documents(all_pdf_documents)
chunks

Split 46 documents into 61 chunks

Example chunk:
content: A practical  
guide to  
building agents...
Metadata: {'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-04-07T14:20:51+00:00', 'moddate': '2025-04-07T14:20:54+00:00', 'source': '..\\data\\pdf\\a-practical-guide-to-building-agents.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1', 'source_file': 'a-practical-guide-to-building-agents.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-04-07T14:20:51+00:00', 'moddate': '2025-04-07T14:20:54+00:00', 'source': '..\\data\\pdf\\a-practical-guide-to-building-agents.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1', 'source_file': 'a-practical-guide-to-building-agents.pdf', 'file_type': 'pdf'}, page_content='A practical \u2028\nguide to \u2028\nbuilding agents'),
 Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-04-07T14:20:51+00:00', 'moddate': '2025-04-07T14:20:54+00:00', 'source': '..\\data\\pdf\\a-practical-guide-to-building-agents.pdf', 'total_pages': 34, 'page': 1, 'page_label': '2', 'source_file': 'a-practical-guide-to-building-agents.pdf', 'file_type': 'pdf'}, page_content='C o n t e n t s\nWha t is an agen t? 4\nWhen should y ou build an agen t